In [ ]:
import requests
import pandas as pd

API_URL = f"https://tabular-api.data.gouv.fr/api/resources/17af574e-ce76-4092-a1f5-f8bbffa307ee/data/"
df_election = pd.read_csv('src/election.csv', sep=';')

all_data = []
page = 1

while True:
    response = requests.get(API_URL, params={"page": page, "page_size": 200})
    result = response.json()
    all_data.extend(result["data"])

    if result["links"]["next"] is None:
        break
    page += 1

df = pd.DataFrame(all_data)
display(df)

In [ ]:
df['gauche'] = df['Hollande'] + df['Mélenchon'] + df['Joly'] + df['Arthaud'] + df['Poutou']
df['centre'] = df['Bayrou']
df['droite'] = df['Sarkozy'] + df['LePen'] + df['DupontAignan'] + df['Cheminade']

df = df.drop(columns=['Hollande', 'Mélenchon', 'Joly', 'Arthaud', 'Poutou', 'Bayrou', 'Sarkozy', 'LePen', 'DupontAignan', 'Cheminade'])

df['Année'] = 2012

df = df.rename(columns={'CodeInsee': 'Code_INSEE'})

display(df)

In [ ]:
df['% Abs/Ins'] = df['Abstentions'] / df['Inscrits'] * 100
df['% Vot/Ins'] = df['Votants'] / df['Inscrits'] * 100

df['% Blancs/Ins'] = df['Blancs'] / df['Inscrits'] * 100
df['% Blancs/Vot'] = df['Blancs'] / df['Votants'] * 100

df['Nuls'] = 0 
df['% Nuls/Ins'] = 0
df['% Nuls/Vot'] = 0

df['% Exp/Ins'] = df['Exprimés'] / df['Inscrits'] * 100
df['% Exp/Vot'] = df['Exprimés'] / df['Votants'] * 100

df['% gauche/Exp'] = df['gauche'] / df['Exprimés'] * 100
df['% centre/Exp'] = df['centre'] / df['Exprimés'] * 100
df['% droite/Exp'] = df['droite'] / df['Exprimés'] * 100

df = df.drop(columns=['gauche', 'centre', 'droite', '__id'])
display(df)

In [ ]:
cols_groups = ['% gauche/Exp', '% centre/Exp', '% droite/Exp']
mask = df[cols_groups].notna().any(axis=1)
df.loc[mask, 'Résultat'] = df.loc[mask, cols_groups].idxmax(axis=1)

groups_dict = {'% gauche/Exp': 'gauche', '% centre/Exp': "centre", '% droite/Exp': 'droite'}
df['Résultat'] = df['Résultat'].replace(groups_dict)


display(df)

In [ ]:
df = df.merge(df_election[['Code_INSEE', 'Libellé de la commune']], on='Code_INSEE', how='left')
df = df.dropna()

In [ ]:
display(df)

In [ ]:
df_election = pd.concat([df_election, df], ignore_index=True)
df_election['Code_INSEE'] = df_election['Code_INSEE'].astype(str)
df_election = df_election.sort_values(by=['Code_INSEE']).reset_index(drop=True)
df_election = df_election.drop_duplicates(subset=['Code_INSEE', 'Année']).reset_index(drop=True)
display(df_election)

df_election.to_csv('src/election.csv', index=False, sep=';', encoding='utf-8')

In [ ]:
df_election.columns